In [ ]:
# Update dependencies and ensure soundfile/librosa are available
!pip install -q "transformers==4.44.2" "datasets==2.21.0" "peft==0.13.0" soundfile librosa pandas-gbq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import random
import re
from typing import List
import numpy as np
import torch
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor
import os

# UPDATED FOR COLAB: Please ensure you have uploaded your model/data or mounted Google Drive
# You can use /content/drive/MyDrive/... if using Google Drive
ADAPTER_PATH = "/content/drive/MyDrive/wisper-model/whisper-medium-quran-lora-multivoice/final_adapter"
DATA_PATH    = "/content/drive/MyDrive/data/processed_quran_ayah_words"

# Helper to check if paths exist and provide guidance
if not os.path.exists(ADAPTER_PATH):
    print(f"WARNING: ADAPTER_PATH not found at {ADAPTER_PATH}. Please upload your adapter files.")
if not os.path.exists(DATA_PATH):
    print(f"WARNING: DATA_PATH not found at {DATA_PATH}. Please upload your dataset.")

# Held-out reciters to calibrate on
HELD_OUT_RECITERS = set()

SAMPLING_RATE = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load base model
base = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-medium",
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

# Load and merge adapter
try:
    model = PeftModel.from_pretrained(base, ADAPTER_PATH).merge_and_unload()
    model = model.to(DEVICE).eval()
    processor = WhisperProcessor.from_pretrained(ADAPTER_PATH)
    print("Model + adapter loaded and merged successfully.")
except Exception as e:
    print(f"Error loading model: {e}")

Model + adapter loaded and merged successfully.


In [ ]:
# Same text-cleaning + minimal-pair code as Notebooks 1/1b/2 (kept in sync).
STRUCTURAL_RE = re.compile(
    "[\u06d6-\u06dc\u06dd\u06de\u06e9\u0610-\u0617\u0640\u0660-\u0669\u06f0-\u06f90-9]"
)

def clean_quranic_text(text: str) -> str:
    text = STRUCTURAL_RE.sub("", text)
    return re.sub(r"\s+", " ", text).strip()

FATHA, DAMMA, KASRA, SUKUN = "\u064e", "\u064f", "\u0650", "\u0652"
FATHATAN, DAMMATAN, KASRATAN = "\u064b", "\u064c", "\u064d"
SHADDAH = "\u0651"
SHORT_HARAKAT = {FATHA, DAMMA, KASRA, SUKUN}
TANWEEN = {FATHATAN, DAMMATAN, KASRATAN}

def harakah_variants(word: str, max_variants: int = None, seed: int = 0) -> List[str]:
    variants = set()
    for i, ch in enumerate(word):
        pool = SHORT_HARAKAT if ch in SHORT_HARAKAT else TANWEEN if ch in TANWEEN else None
        if pool is not None:
            for alt in pool - {ch}:
                variants.add(word[:i] + alt + word[i + 1:])
        elif ch == SHADDAH:
            variants.add(word[:i] + word[i + 1:])
    variants.discard(word)
    out = sorted(variants)
    if max_variants is not None and len(out) > max_variants:
        out = random.Random(seed).sample(out, max_variants)
    return out

In [ ]:
def forced_score(input_features: torch.Tensor, texts: List[str]):
    """Teacher-force each candidate text against the SAME audio.
    Returns (mean_logprob_per_text, per_token_detail_for_text_0)."""
    tok = processor.tokenizer
    pad_id = tok.pad_token_id or tok.eos_token_id
    seqs = [tok(t).input_ids for t in texts]
    maxlen = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), maxlen), pad_id, dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
    decoder_input_ids = ids[:, :-1].to(DEVICE)
    targets = ids[:, 1:].to(DEVICE)

    with torch.no_grad():
        logits = model(
            input_features=input_features.expand(len(seqs), -1, -1),
            decoder_input_ids=decoder_input_ids,
        ).logits
    token_logprobs = logits.float().log_softmax(-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1)

    mask = torch.zeros_like(targets, dtype=torch.bool)
    for i, s in enumerate(seqs):
        mask[i, 3:len(s) - 2] = True
    scores = (token_logprobs * mask).sum(1) / mask.sum(1).clamp(min=1)

    detail = [
        (tok.decode([targets[0, j].item()]), round(token_logprobs[0, j].item(), 3))
        for j in range(3, len(seqs[0]) - 2)
    ]
    return scores, detail


def analyze_word(audio_array: np.ndarray, expected_word: str,
                 margin_threshold: float = 0.0) -> dict:
    """The app per-word verdict. margin_threshold: minimum lead the correct form
    must have over the best wrong variant to pass. CALIBRATE ON REAL USER AUDIO."""
    expected = clean_quranic_text(expected_word)
    feats = processor.feature_extractor(
        np.asarray(audio_array, dtype=np.float32),
        sampling_rate=SAMPLING_RATE, return_tensors="pt",
    ).input_features.to(DEVICE).half()

    variants = harakah_variants(expected)
    candidates = [expected] + variants
    scores, detail = forced_score(feats, candidates)
    scores = scores.tolist()

    ranked = sorted(zip(candidates, scores), key=lambda x: -x[1])
    best_wrong = max(scores[1:]) if variants else float("-inf")
    margin = scores[0] - best_wrong

    return {
        "expected": expected,
        "verdict": "pass" if margin > margin_threshold else "flag",
        "margin": round(margin, 4),
        "expected_score": round(scores[0], 4),
        "most_likely_said": ranked[0][0],
        "top_candidates": [(w, round(s, 4)) for w, s in ranked[:5]],
        "per_token_confidence": detail,
    }

In [ ]:
from datasets import load_from_disk

ds = load_from_disk(DATA_PATH)
reciters = np.array(ds["reciter"])
# Demo on an unseen-reciter word if a held-out set is configured, else any word.
if HELD_OUT_RECITERS:
    pool = np.where(np.isin(reciters, list(HELD_OUT_RECITERS)))[0]
else:
    pool = np.arange(len(ds))
sample = ds[int(pool[10])]
audio, word = sample["audio_array"], sample["text"]
print("Word under test:", word, "| reciter:", sample["reciter"])

print("\n--- 1) Correct recitation vs correct expected text ---")
result = analyze_word(audio, word)
for k, v in result.items():
    print(f"{k}: {v}")

print("\n--- 2) Same audio vs a WRONG expected text (simulated learner error) ---")
wrong_expected = harakah_variants(word, max_variants=1)[0]
result = analyze_word(audio, wrong_expected)
for k, v in result.items():
    print(f"{k}: {v}")
print("\nExpected: verdict='flag' and most_likely_said ==", word)

Word under test: فَلْيَتَّقُوا | reciter: minshawy_teacher

--- 1) Correct recitation vs correct expected text ---


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


expected: فَلْيَتَّقُوا
verdict: flag
margin: -0.3324
expected_score: -3.2329
most_likely_said: فَلْيَتَقُوا
top_candidates: [('فَلْيَتَقُوا', -2.9005), ('فِلْيَتَّقُوا', -2.9271), ('فُلْيَتَّقُوا', -3.0639), ('فَلْيَتَّقُوا', -3.2329), ('فَلْيَتَّقَوا', -3.2949)]
per_token_confidence: [('ل', -3.281), ('ْ', -0.656), ('ي', -0.22), ('َ', -2.82), ('ت', -6.48), ('ّ', -2.645), ('َ', -5.411), ('ق', -4.602), ('ُ', -3.182), ('وا', -3.033)]

--- 2) Same audio vs a WRONG expected text (simulated learner error) ---
expected: فَلْيْتَّقُوا
verdict: flag
margin: -1.9064
expected_score: -5.1393
most_likely_said: فَلْيَتَّقُوا
top_candidates: [('فَلْيَتَّقُوا', -3.2329), ('فَلْيُتَّقُوا', -3.3616), ('فَلْيْتَقُوا', -3.8197), ('فَلْيِتَّقُوا', -4.002), ('فَلِيْتَّقُوا', -4.7421)]
per_token_confidence: [('ل', -3.281), ('ْ', -0.656), ('ي', -0.22), ('ْ', -11.974), ('ت', -7.211), ('ّ', -8.877), ('َ', -7.165), ('ق', -5.217), ('ُ', -3.765), ('وا', -3.026)]

Expected: verdict='flag' and most_likely_said == ف

In [ ]:
# Calibration: score held-out-RECITER words (unseen voices) twice each --
# against the correct text (want pass) and a perturbed text (want flag).
N_CALIB = 300

ds = load_from_disk(DATA_PATH)
reciters = np.array(ds["reciter"])
if HELD_OUT_RECITERS:
    held_idx = np.where(np.isin(reciters, list(HELD_OUT_RECITERS)))[0]
else:
    print("WARNING: HELD_OUT_RECITERS empty -> calibrating on a random slice "
          "(does NOT measure unseen-voice accuracy). Set it to match NB2.")
    rng0 = np.random.default_rng(0)
    held_idx = np.where(rng0.random(len(ds)) < 0.05)[0]

rng = np.random.default_rng(0)
if N_CALIB is not None and len(held_idx) > N_CALIB:
    held_idx = rng.choice(held_idx, size=N_CALIB, replace=False)
print(f"Calibrating on {len(held_idx)} held-out-reciter words...")

correct_margins, wrong_margins, n_skipped = [], [], 0
for n, i in enumerate(held_idx):
    row = ds[int(i)]
    audio, true_word = row["audio_array"], row["text"]
    perturbations = harakah_variants(clean_quranic_text(true_word), max_variants=1)
    if not perturbations:
        n_skipped += 1
        continue
    correct_margins.append(analyze_word(audio, true_word)["margin"])
    wrong_margins.append(analyze_word(audio, perturbations[0])["margin"])
    if (n + 1) % 50 == 0:
        print(f"  scored {n + 1}/{len(held_idx)}")

correct_margins = np.array(correct_margins)
wrong_margins   = np.array(wrong_margins)
print(f"Done. {len(correct_margins)} usable words ({n_skipped} skipped).")

Calibrating on 300 held-out-reciter words...
  scored 50/300
  scored 100/300
  scored 150/300
  scored 200/300
  scored 250/300
  scored 300/300
Done. 300 usable words (0 skipped).


In [ ]:
candidates = np.unique(np.concatenate([correct_margins, wrong_margins]))
sweep = np.linspace(candidates.min(), candidates.max(), 200)

best_t, best_bal = 0.0, -1.0
for t in sweep:
    pass_rate  = (correct_margins > t).mean()
    catch_rate = (wrong_margins  <= t).mean()
    bal = 0.5 * (pass_rate + catch_rate)
    if bal > best_bal:
        best_bal, best_t = bal, t

def report(t):
    pr = (correct_margins > t).mean()
    cr = (wrong_margins  <= t).mean()
    return pr, cr, 0.5 * (pr + cr)

pr0, cr0, bal0 = report(0.0)
prb, crb, balb = report(best_t)

print("=" * 64)
print(f"Held-out words scored : {len(correct_margins)}")
print(f"Correct-word margins  : mean {correct_margins.mean():+.3f}  median {np.median(correct_margins):+.3f}")
print(f"Mistake margins       : mean {wrong_margins.mean():+.3f}  median {np.median(wrong_margins):+.3f}")
print("-" * 64)
print(f"{'threshold':>22} | {'pass-rate':>10} | {'catch-rate':>11} | {'balanced':>9}")
print(f"{'0.000 (naive)':>22} | {pr0:>9.1%} | {cr0:>10.1%} | {bal0:>8.1%}")
print(f"{best_t:>15.3f} (best) | {prb:>9.1%} | {crb:>10.1%} | {balb:>8.1%}")
print("=" * 64)
print(f"\n>>> RECOMMENDED margin_threshold = {best_t:.3f}")
print(f"    pass-rate={prb:.1%}  catch-rate={crb:.1%}")
print("\nBaseline to beat (single-voice model): ~80% pass / ~77% catch / ~78% balanced.")
print("Re-calibrate on REAL user audio before shipping.")

print("\nMargin distributions (C=correct, X=mistake):")
bins = np.linspace(candidates.min(), candidates.max(), 28)
ch, _ = np.histogram(correct_margins, bins=bins)
xh, _ = np.histogram(wrong_margins,  bins=bins)
scale = max(ch.max(), xh.max()) / 30 or 1
for b in range(len(bins) - 1):
    mid = (bins[b] + bins[b + 1]) / 2
    marker = " <- threshold" if bins[b] <= best_t < bins[b + 1] else ""
    print(f"{mid:+6.2f} | C {'#' * int(ch[b] / scale):<31}{marker}")
    print(f"       | X {'#' * int(xh[b] / scale):<31}")

Held-out words scored : 300
Correct-word margins  : mean -0.376  median -0.081
Mistake margins       : mean -1.821  median -1.370
----------------------------------------------------------------
             threshold |  pass-rate |  catch-rate |  balanced
         0.000 (naive) |     45.7% |      93.7% |    69.7%
         -0.697 (best) |     74.0% |      76.3% |    75.2%

>>> RECOMMENDED margin_threshold = -0.697
    pass-rate=74.0%  catch-rate=76.3%

Baseline to beat (single-voice model): ~80% pass / ~77% catch / ~78% balanced.
Re-calibrate on REAL user audio before shipping.

Margin distributions (C=correct, X=mistake):
-11.74 | C                                
       | X                                
-11.12 | C                                
       | X                                
-10.51 | C                                
       | X                                
 -9.89 | C                                
       | X                                
 -9.28 | C               